In [3]:
import time
from contextlib import contextmanager
@contextmanager
def timer(label=""):
    start = time.perf_counter()
    print(f"[{label}] 开始计时...")
    try:
        yield  # yield 之前 = __enter__，之后 = __exit__
    finally:
        elapsed = (time.perf_counter() - start) * 1000
        print(f"[{label}] 耗时: {elapsed:.3f}ms")
with timer("数据处理"):
    total = sum(range(1_000_000))
    print(f"  计算结果: {total}")
print()
try:
    with timer("异常测试"):
        print("  即将出错...")
        raise RuntimeError("boom")
except RuntimeError:
    print("  异常被外部捕获，但计时器仍然执行了 finally")

[数据处理] 开始计时...
  计算结果: 499999500000
[数据处理] 耗时: 18.414ms

[异常测试] 开始计时...
  即将出错...
[异常测试] 耗时: 0.023ms
  异常被外部捕获，但计时器仍然执行了 finally


In [3]:
from contextlib import contextmanager
class MockDBConnection:
    _pool = []
    def __init__(self,conn_id):
        self.conn_id = conn_id
        self.in_transaction = False
    def execute(self,sql):
        print(f"  [conn-{self.conn_id}] 执行: {sql}")
        self.in_transaction = True
    def commit(self):
        print(f"  [conn-{self.conn_id}] 提交事务")
        self.in_transaction = False
    def rollback(self):
        print(f"  [conn-{self.conn_id}] 回滚事务")
        self.in_transaction = False
    def close(self):
        print(f"  [conn-{self.conn_id}] 关闭连接，归还连接池")
@contextmanager
def db_transaction(conn):
    try:
        yield conn
        conn.commit()
    except Exception as e:
        conn.rollback()
        raise
    finally:
        conn.close()
conn = MockDBConnection(1)
with db_transaction(conn) as c:
    c.execute("INSERT INTO users VALUES ('Alice', 30)")
    c.execute("INSERT INTO orders VALUES (1, 'MacBook')")
print()

  [conn-1] 执行: INSERT INTO users VALUES ('Alice', 30)
  [conn-1] 执行: INSERT INTO orders VALUES (1, 'MacBook')
  [conn-1] 提交事务
  [conn-1] 关闭连接，归还连接池



In [6]:
from contextlib import suppress,redirect_stdout,closing
import io
with suppress(FileNotFoundError):
    import os
    os.remove("/tmp/nonexistent_file.txt")
print("suppress: 文件不存在也不报错")
buffer = io.StringIO()
with redirect_stdout(buffer):
    print("这行不会显示在终端")
    print("而是被捕获到 buffer 中")
captured = buffer.getvalue()
print(captured)
class Resource:
    def __init__(self,name):
        self.name = name
    def close(self):
        print(f"  [{self.name}] 资源已关闭")
    def use(self):
        print(f"  [{self.name}] 正在使用...")
with closing(Resource("数据库连接")) as res:
    res.use()

suppress: 文件不存在也不报错
这行不会显示在终端
而是被捕获到 buffer 中

  [数据库连接] 正在使用...
  [数据库连接] 资源已关闭


In [7]:
from contextlib import ExitStack,contextmanager
@contextmanager
def managed_resource(name):
    print(f"  [acquire] {name}")
    try:
        yield name
    finally:
        print(f"  [release] {name}")
resource_names = ['数据库','缓存','消息队列','日志服务']
with ExitStack() as stack:
    resources = [
        stack.enter_context(managed_resource(name))
        for name in resource_names
    ]
    print(f"  所有资源已就绪: {resources}")
    if len(resources) > 2:
        print("  模拟异常！")
        raise RuntimeError("服务不可用")

  [acquire] 数据库
  [acquire] 缓存
  [acquire] 消息队列
  [acquire] 日志服务
  所有资源已就绪: ['数据库', '缓存', '消息队列', '日志服务']
  模拟异常！
  [release] 日志服务
  [release] 消息队列
  [release] 缓存
  [release] 数据库


RuntimeError: 服务不可用

In [8]:
def demo(value):
    try:
        if value == 1:
            raise ValueError("值错误")
        elif value == 2:
            raise TypeError("类型错误")
        elif value == 3:
            raise OSError(2,"文件不存在", "test.txt")
        elif value == 4:
            raise KeyboardInterrupt
    except ValueError as e:
        print(f"  捕获 ValueError: {e}")
    except (TypeError,OSError) as e:
        print(f"  捕获 TypeError/OSError: {type(e).__name__}: {e}")
    except Exception as e:
        print(f"  捕获 Exception: {type(e).__name__}: {e}")
    except BaseException as e:
        print(f"  捕获 BaseException: {type(e).__name__}")
for i in range(1,5):
    print(f"--- value={i} ---")
    try:
        demo(i)
    except BaseException:
        print(f"  外层兜底捕获")

--- value=1 ---
  捕获 ValueError: 值错误
--- value=2 ---
  捕获 TypeError/OSError: TypeError: 类型错误
--- value=3 ---
  捕获 TypeError/OSError: FileNotFoundError: [Errno 2] 文件不存在: 'test.txt'
--- value=4 ---
  捕获 BaseException: KeyboardInterrupt


In [9]:
class OrderError(Exception):
    def __init__(self,message,order_id=None):
        super().__init__()
        self.order_id = order_id
class InsufficientStockError(OrderError):
    def __init__(self,product_id,requested,available,order_id=None):
        super().__init__(
            f"商品{product_id}库存不足: 需要{requested}, 剩余{available}",
            order_id=order_id
        )
        self.product_id = product_id
        self.requested = requested
        self.available = available
class PaymentFailedError(OrderError):
    def __init__(self,reason,order_id=None):
        super().__init__(f"支付失败: {reason}", order_id=order_id)
        self.reason = reason
class OrderCancelledError(OrderError):
    """订单已取消"""
    pass
def process_order(order_id,items):
    try:
        raise InsufficientStockError("SKU-001", 5, 2, order_id)
    except OrderError as e:
        print(f"订单{e.order_id}处理失败: {e}")
        print(f"  错误类型: {type(e).__name__}")
        if isinstance(e,InsufficientStockError):
            print(f"  建议: 通知用户商品{e.product_id}库存仅剩{e.available}件")
        elif isinstance(e,PaymentFailedError):
            print(f"  建议: 引导用户更换支付方式")
process_order("ORD-20240101-001", [{"sku": "SKU-001", "qty": 5}])

订单ORD-20240101-001处理失败: 
  错误类型: InsufficientStockError
  建议: 通知用户商品SKU-001库存仅剩2件
